# CSP-10 — Couverture minimale avec Z3 en C#

[<< CSP-9](CSP-9-Distributed-CSharp.ipynb) | [Index CSP](README.md) | [Version Python](CSP-10-Optimisation-Z3.ipynb)

**Durée indicative : 1 h 30.** Ce binôme du notebook Python formule une couverture minimale, compare Z3 à une énumération exacte sur une petite instance et démontre une borne inférieure par insatisfaisabilité. Prérequis : CSP-1, CSP-2 et CSP-5, ainsi que C# et .NET Interactive.

La couverture d'ensembles générale est NP-complète sous sa forme de décision. Ici chaque case d'une grille est un capteur candidat qui couvre le carré 3×3 centré sur elle, tronqué au bord. On cherche le minimum de capteurs dont l'union couvre toutes les cases. C'est un ensemble dominant dans le graphe où deux cases sont voisines si leurs coordonnées diffèrent d'au plus un sur chaque axe (grille avec diagonales).

### Installation et moteur

Le paquet NuGet **Microsoft.Z3 4.12.2** fournit l'API .NET ; il ne livre la bibliothèque native que pour Windows x64 et macOS Intel. Ailleurs (Linux, macOS Apple Silicon), la cellule d'amorce s'appuie sur le chargeur partagé `Z3NativeLoader.cs` de la série Z3-API : il prend la bibliothèque du paquet Python `z3-solver` qu'installe `pip install -r MyIA.AI.Notebooks/Search/requirements.txt`, sans variable d'environnement à poser (`Z3_LIBRARY_PATH` permet d'en désigner une autre). Pour aligner la version native sur celle de l'API .NET, installer `z3-solver==4.12.2.0` ; la cardinalité optimale et la preuve de borne n'en dépendent pas, le placement optimal retenu peut en dépendre.

Le notebook n'utilise Python que pour localiser cette bibliothèque native : toute la modélisation et les appels solveur sont en C#. Exécuter depuis ce dossier avec le kernel `.net-csharp` ; une restauration NuGet avec accès réseau est nécessaire au premier lancement.

In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
Console.WriteLine("Paquet Microsoft.Z3 4.12.2 restauré.");

Installed Packages Microsoft.Z3, 4.12.2

Paquet Microsoft.Z3 4.12.2 restauré.


Le paquet .NET est restauré. La cellule suivante enregistre le chargeur de la bibliothèque native décrit en tête du carnet, avant le premier appel à Z3 ; sous Windows x64 et macOS Intel, elle laisse le paquet NuGet fournir la sienne.

In [2]:
#load "../../SymbolicAI/SMT/Z3-API/Z3NativeLoader.cs"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Numerics;
using System.Runtime.InteropServices;
using Microsoft.Z3;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine($"Moteur natif : {Microsoft.Z3.Version.FullVersion}; plateforme : {RuntimeInformation.OSDescription}; architecture : {RuntimeInformation.ProcessArchitecture}");

Moteur natif : 5.4.1.0; plateforme : Ubuntu 24.04.5 LTS; architecture : X64


## 1. Construire l'instance

Pour chaque point, on calcule les indices des capteurs susceptibles de le couvrir. Une grille 4×3 contient 12 capteurs et $2^{12}$ sous-ensembles possibles.

In [3]:
public class Grille
{
    public int Lignes, Colonnes;
    public (int r, int c)[] Points;
    public int[][] Couvreurs;
    public Grille(int lignes, int colonnes)
    {
        Lignes = lignes; Colonnes = colonnes;
        Points = Enumerable.Range(0, lignes * colonnes).Select(i => (i / colonnes, i % colonnes)).ToArray();
        Couvreurs = Points.Select(p => Enumerable.Range(0, Points.Length)
            .Where(i => Math.Abs(Points[i].r - p.r) <= 1 && Math.Abs(Points[i].c - p.c) <= 1).ToArray()).ToArray();
    }
    // Contrôle indépendant : coordonnées géométriques, sans réutiliser Couvreurs.
    public bool Valide(int[] choix) => Points.All(p => choix.Any(i =>
        Math.Abs(Points[i].r - p.r) <= 1 && Math.Abs(Points[i].c - p.c) <= 1));
}
var petite = new Grille(4, 3);
Console.WriteLine($"Petite grille : {petite.Points.Length} points et capteurs ; {BigInteger.Pow(2, petite.Points.Length)} sous-ensembles possibles.");

Petite grille : 12 points et capteurs ; 4096 sous-ensembles possibles.


### Exemple : oracle exhaustif

On énumère les masques par cardinalité croissante et on s'arrête à la première couverture. L'énumération a un coût exponentiel dans le pire cas ; elle sert d'oracle seulement pour la petite grille.

In [4]:
int[] Oracle(Grille g)
{
    int n = g.Points.Length;
    for (int k = 0; k <= n; k++)
        for (int masque = 0; masque < (1 << n); masque++)
            if (BitOperations.PopCount((uint)masque) == k)
            {
                var choix = Enumerable.Range(0, n).Where(i => (masque & (1 << i)) != 0).ToArray();
                if (g.Valide(choix)) return choix;
            }
    return Array.Empty<int>();
}
var chronoBrut = Stopwatch.StartNew();
var brut = Oracle(petite); chronoBrut.Stop();
Console.WriteLine($"Oracle : {brut.Length} capteurs [{string.Join(", ", brut)}], couverture {petite.Valide(brut)}, temps {chronoBrut.Elapsed.TotalMilliseconds:F3} ms.");

Oracle : 2 capteurs [1, 7], couverture True, temps 4.063 ms.


L'oracle a contrôlé géométriquement la couverture et parcouru les tailles dans l'ordre : la première taille trouvée est minimale. La durée affichée dépend de la machine et ne doit pas être extrapolée à la grande grille.

### Exercice 1 — Budget de capteurs

Sur la grande grille construite plus bas, imposez $\sum x_i \le K$ avec un `Solver` et cherchez le plus petit budget admissible. Comparez-le à l'optimum Z3. Le squelette retourne une valeur absente tant que l'exercice n'est pas complété.

In [5]:
int? BudgetMinimal(Grille grille) {
    // TODO : construire un Solver, tester les budgets croissants et retourner le premier seuil admissible.
    return null;
}
Console.WriteLine(BudgetMinimal(petite)?.ToString() ?? "Exercice 1 à compléter : aucun budget calculé.");

Exercice 1 à compléter : aucun budget calculé.


## 2. Optimisation SMT

Chaque capteur est un booléen $x_i$. Pour chaque point, la disjonction des variables des capteurs qui le couvrent doit être vraie. `Optimize` minimise $\sum_i \operatorname{ite}(x_i,1,0)$. Un second `Solver` impose une cardinalité strictement inférieure à celle du modèle. Seul `UNSATISFIABLE` prouve la minimalité ; `UNKNOWN` ou un échec ne la prouvent pas.

Pour accélérer `Optimize`, on ajoute une borne inférieure redondante : les points de coordonnées multiples de trois ne peuvent être couverts deux à deux par un même capteur. Le `Solver` de preuve ne reçoit pas cette borne ; il reprend uniquement la couverture et une contrainte de cardinalité stricte.

In [6]:
public record Resultat(int[] Choix, int Optimum, Status Optimisation, Status Borne, double Millisecondes);
Resultat Resoudre(Grille g)
{
    var sw = Stopwatch.StartNew();
    using var ctx = new Context();
    var x = Enumerable.Range(0, g.Points.Length).Select(i => ctx.MkBoolConst($"x{i}")).ToArray();
    var contraintes = g.Couvreurs.Select(indices => ctx.MkOr(indices.Select(i => x[i]).ToArray())).ToArray();
    var cout = ctx.MkAdd(x.Select(v => (ArithExpr)ctx.MkITE(v, ctx.MkInt(1), ctx.MkInt(0))).ToArray());
    using var opt = ctx.MkOptimize();
    opt.Assert(constraints: contraintes);
    // Points espacés de trois : aucun capteur ne couvre deux de ces points.
    int temoins = ((g.Lignes + 2) / 3) * ((g.Colonnes + 2) / 3);
    opt.Assert(ctx.MkGe(cout, ctx.MkInt(temoins)));
    opt.MkMinimize(cout);
    var statut = opt.Check();
    if (statut != Status.SATISFIABLE) throw new Exception($"Optimisation non résolue : {statut}");
    var modele = opt.Model;
    var choix = Enumerable.Range(0, x.Length).Where(i => modele.Evaluate(x[i], true).IsTrue).ToArray();
    int valeur = ((IntNum)modele.Evaluate(cout, true)).Int;
    if (choix.Length != valeur || !g.Valide(choix)) throw new Exception("Modèle non valide géométriquement");
    using var preuve = ctx.MkSolver();
    preuve.Assert(contraintes);
    preuve.Assert(ctx.MkAtMost(x, (uint)(valeur - 1)));
    var borne = preuve.Check();
    sw.Stop();
    return new Resultat(choix, valeur, statut, borne, sw.Elapsed.TotalMilliseconds);
}
var zpetit = Resoudre(petite);
Console.WriteLine($"Optimize : {zpetit.Optimisation}, optimum {zpetit.Optimum}, indices [{string.Join(", ", zpetit.Choix)}]");
Console.WriteLine($"Couverture indépendante : {petite.Valide(zpetit.Choix)} ; Solver cardinalité < {zpetit.Optimum} : {zpetit.Borne} ; accord oracle : {zpetit.Optimum == brut.Length}");

Optimize : SATISFIABLE, optimum 2, indices [1, 10]


Couverture indépendante : True ; Solver cardinalité < 2 : UNSATISFIABLE ; accord oracle : True


L'accord avec l'oracle porte sur la cardinalité, pas nécessairement sur les indices : plusieurs placements optimaux peuvent exister. La vérification géométrique contrôle la faisabilité ; le statut du second solveur contrôle séparément la borne inférieure.

### Exercice 2 — Portées hétérogènes

Sur la petite grille, mélangez des capteurs carrés 3×3 et d'autres en croix (centre et quatre voisins orthogonaux). Construisez pour chaque point les capteurs qui le couvrent, résolvez le modèle et comparez les optima. Le contrôle géométrique devra lui aussi connaître les deux formes.

In [7]:
int[][] CouvreursMixtes(Grille grille) {
    // TODO : construire les listes pour un choix explicite de formes par capteur.
    return null;
}
Console.WriteLine(CouvreursMixtes(petite) == null ? "Exercice 2 à compléter : aucune instance mixte calculée." : "Instance mixte construite.");

Exercice 2 à compléter : aucune instance mixte calculée.


## 3. Grande grille et passage à l'échelle

Sur 10×6, il y a 60 candidats et $2^{60}$ sous-ensembles possibles. Ce nombre est l'espace de recherche brut, pas une prédiction exacte du travail de Z3 ; aucune énumération exhaustive n'est lancée sur cette instance.

In [8]:
var grande = new Grille(10, 6);
var zgrand = Resoudre(grande);
Console.WriteLine($"Grande grille : {grande.Points.Length} cases, {BigInteger.Pow(2, grande.Points.Length)} sous-ensembles possibles.");
Console.WriteLine($"Optimize : {zgrand.Optimisation}, optimum {zgrand.Optimum}, indices [{string.Join(", ", zgrand.Choix)}]");
Console.WriteLine($"Couverture indépendante : {grande.Valide(zgrand.Choix)} ; Solver cardinalité < {zgrand.Optimum} : {zgrand.Borne} ; optimisation et preuve : {zgrand.Millisecondes:F3} ms.");

Grande grille : 60 cases, 1152921504606846976 sous-ensembles possibles.


Optimize : SATISFIABLE, optimum 8, indices [7, 10, 25, 28, 37, 46, 55, 58]


Couverture indépendante : True ; Solver cardinalité < 8 : UNSATISFIABLE ; optimisation et preuve : 2374.156 ms.


Le statut et le contrôle de couverture affichés permettent de distinguer une solution réalisable d'un optimum démontré. La durée mesure ici l'optimisation et le second appel, sur cette machine seulement ; elle ne constitue pas une garantie de performance générale.

In [9]:
string Dessiner(Grille g, int[] choix)
{
    int pas = 42, marge = 28;
    var s = new System.Text.StringBuilder();
    s.Append($"<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Grille couverte et capteurs sélectionnés' viewBox='0 0 {g.Colonnes*pas+2*marge} {g.Lignes*pas+2*marge}' width='380'>");
    s.Append("<rect width='100%' height='100%' fill='white'/>");
    foreach (var p in g.Points) s.Append($"<circle cx='{marge+p.c*pas}' cy='{marge+p.r*pas}' r='12' fill='#b8e0c2' stroke='#285e3b'/>");
    foreach (var i in choix) { var p=g.Points[i]; s.Append($"<circle cx='{marge+p.c*pas}' cy='{marge+p.r*pas}' r='7' fill='#bc3030'/>"); }
    s.Append("</svg><p>Vert : point couvert ; rouge : capteur sélectionné.</p>");
    return s.ToString();
}
public record PlotSvg(string Content);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<PlotSvg>((v, w) => w.Write(v.Content), "text/html");
display(new PlotSvg(Dessiner(grande, zgrand.Choix)));

Vert : point couvert ; rouge : capteur sélectionné.

Le tracé représente les capteurs retenus et les points couverts. Le choix du modèle peut différer d'une exécution à l'autre lorsque plusieurs solutions optimales existent.

### Exercice 3 — Prouver la borne inférieure

Construisez un nouveau `Solver` sans `Optimize`, imposez la couverture et $\sum x_i \le m-1$, où $m$ est l'optimum de la petite grille. Interprétez `UNSATISFIABLE` et expliquez pourquoi `UNKNOWN` ne suffit pas. Comparez votre statut à celui du solveur de preuve ci-dessus.

In [10]:
Status? ProuverBorne(Grille grille, int m) {
    // TODO : créer Solver et variables, ajouter couverture et borne, puis retourner Check().
    return null;
}
Console.WriteLine(ProuverBorne(petite, zpetit.Optimum)?.ToString() ?? "Exercice 3 à compléter : aucune preuve produite.");

Exercice 3 à compléter : aucune preuve produite.


## Conclusion et références

Le modèle déclaratif relie CP, SAT et SMT : les contraintes de couverture restent les mêmes alors que le moteur change. L'oracle exhaustif donne un témoin indépendant sur la petite grille ; `Optimize`, la validation géométrique et le test de borne par `Solver` jouent trois rôles complémentaires sur les deux grilles.

- de Moura et Bjørner (2008), « Z3: An Efficient SMT Solver », TACAS, LNCS 4956, 357–371.
- Garey et Johnson (1979), *Computers and Intractability*, W. H. Freeman.
- Bardin, Bjørner et Cadar (2019), « Bringing CP, SAT and SMT together », [Dagstuhl Reports 9(2):27](https://doi.org/10.4230/DagRep.9.2.27).